In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.shape

(768, 9)

In [6]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [7]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [9]:
X = scaler.fit_transform(X)

In [10]:
X.shape

(768, 8)

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state= 1
)

In [12]:
import tensorflow as tf
from tensorflow import keras 
from keras import Sequential 
from keras.layers import Dense

In [13]:
model = Sequential()
model.add(Dense (32, activation= 'relu', input_dim = 8))
model.add(Dense(1, activation= 'sigmoid'))

In [14]:
model.compile(optimizer= 'Adam', loss = 'binary_crossentropy', metrics= ['accuracy'])

In [15]:
model.fit(X_train, y_train, batch_size= 32, epochs= 100, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6564 - loss: 0.6769 - val_accuracy: 0.6688 - val_loss: 0.6445
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6873 - loss: 0.6254 - val_accuracy: 0.7273 - val_loss: 0.5999
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7166 - loss: 0.5896 - val_accuracy: 0.7403 - val_loss: 0.5702
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7182 - loss: 0.5626 - val_accuracy: 0.7532 - val_loss: 0.5469
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7215 - loss: 0.5419 - val_accuracy: 0.7922 - val_loss: 0.5278
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7248 - loss: 0.5258 - val_accuracy: 0.7987 - val_loss: 0.5140
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7443 - loss: 0.5129 - val_accuracy: 0.8052 - val_loss: 0.5031
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7524 - loss: 0.5009 - val_accuracy: 0.7922 - v

In [16]:
import kerastuner as kt

In [17]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(32, activation= 'relu', input_dim = 8))
    model.add(Dense(1, activation='sigmoid'))
    optimizer= hp.Choice('optimizer', values= ['adam', 'sgd', 'rmsprop', 'adadelta',])

    model.compile(optimizer = optimizer,
                  loss = 'binary_crossentropy', metrics= ['accuracy'])

    return model

In [18]:
tuner = kt.RandomSearch(build_model,
                        objective= 'val_accuracy',
                        max_trials= 5)

Reloading Tuner from .\untitled_project\tuner0.json


In [19]:
tuner.search(X_train, y_train, epochs = 5,validation_data= (X_test, y_test))

In [20]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [21]:
tuner.results_summary()

Results summary
Results in .\untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.7272727489471436

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7142857313156128

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.6298701167106628

Trial 1 summary
Hyperparameters:
optimizer: adadelta
Score: 0.38311687111854553


In [22]:
model = tuner.get_best_models(num_models= 1)[0] 

In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(X_train, y_train, batch_size= 32, epochs= 100, initial_epoch= 6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7280 - loss: 0.5848 - val_accuracy: 0.7532 - val_loss: 0.5706
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7427 - loss: 0.5596 - val_accuracy: 0.7727 - val_loss: 0.5498
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7508 - loss: 0.5395 - val_accuracy: 0.7792 - val_loss: 0.5335
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7622 - loss: 0.5229 - val_accuracy: 0.7857 - val_loss: 0.5194
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7606 - loss: 0.5097 - val_accuracy: 0.7922 - val_loss: 0.5106
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7622 - loss: 0.4994 - val_accuracy: 0.7857 - val_loss: 0.5015
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7671 - loss: 0.4905 - val_accuracy: 0.8052 - val_loss: 0.4933
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7720 - loss: 0.4826 - val_accuracy: 0.798

In [25]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

def build_model(hp):

    model = Sequential()

    # Number of hidden layers
    num_layers = hp.Int('num_layers', min_value=1, max_value=5)

    for i in range(num_layers):

        # Number of neurons
        units = hp.Int(
            f'units_{i}',
            min_value=8,
            max_value=128,
            step=8
        )

        # Activation function
        activation = hp.Choice(
            f'activation_{i}',
            values=['relu', 'tanh', 'elu']
        )

        model.add(Dense(
            units=units,
            activation=activation,
            input_dim=8 if i == 0 else None
        ) if i == 0 else Dense(
            units=units,
            activation=activation
        ))

        # Dropout rate
        dropout_rate = hp.Choice(
            f'dropout_{i}',
            values=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
        )

        model.add(Dropout(dropout_rate))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Optimizer
    optimizer = hp.Choice(
        'optimizer',
        values=['adam', 'rmsprop', 'sgd', 'nadam', 'adadelta']
    )

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [26]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    overwrite=True,
    directory='tuning',
    project_name='diabetes_tuning'
)

In [27]:
tuner.search(X_train, y_train, epochs= 5, validation_data = (X_test, y_test))

Trial 10 Complete [00h 00m 02s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 21s


In [28]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4,
 'units_0': 56,
 'activation_0': 'tanh',
 'dropout_0': 0.1,
 'optimizer': 'adam',
 'units_1': 96,
 'activation_1': 'elu',
 'dropout_1': 0.3,
 'units_2': 96,
 'activation_2': 'elu',
 'dropout_2': 0.2,
 'units_3': 8,
 'activation_3': 'relu',
 'dropout_3': 0.0}

In [29]:
model.fit(X_train, y_train, epochs=200, initial_epoch= 5, validation_data=(X_test, y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4186 - val_accuracy: 0.8182 - val_loss: 0.4590
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8029 - loss: 0.4187 - val_accuracy: 0.8182 - val_loss: 0.4590
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8029 - loss: 0.4183 - val_accuracy: 0.8247 - val_loss: 0.4610
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8013 - loss: 0.4176 - val_accuracy: 0.8247 - val_loss: 0.4602
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8029 - loss: 0.4174 - val_accuracy: 0.8182 - val_loss: 0.4598
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8013 - loss: 0.4171 - val_accuracy: 0.8182 - val_loss: 0.4608
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7997 - loss: 0.4167 - val_accuracy: 0.8247 - val_loss: 0.4605
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8013 - loss: 0.4164 - val_accuracy: 0.8182